In [2]:
# ==========================================================
# Project: Auto Tagging Support Tickets Using LLM
# Objective:
# Automatically classify support tickets into relevant
# categories using Zero-Shot and Prompt-Engineered approaches.
#
# Model Used:
# facebook/bart-large-mnli (Zero-Shot Classification Model)
#
# The system outputs Top 3 predicted tags with confidence scores.
# Performance is evaluated using accuracy metric.
# ==========================================================
# Install required libraries
!pip install transformers
!pip install torch
!pip install pandas
!pip install scikit-learn

In [3]:
#Importing Libraries
import pandas as pd
import numpy as np
from transformers import pipeline
from sklearn.metrics import accuracy_score

In [4]:
#Load Zero-Shot Model
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
# ----------------------------------------------------------
# Load Zero-Shot Classification Model
# ----------------------------------------------------------
# We use facebook/bart-large-mnli which is trained on
# Natural Language Inference (NLI).
# It allows us to classify text into categories without
# training on our specific dataset (Zero-Shot Learning).
# ----------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#Create Dataset
data = {
    "ticket": [
        "I cannot log into my account",
        "Payment was deducted but order not confirmed",
        "App crashes when I open settings",
        "How can I reset my password?",
        "The website is very slow today",
        "I want to cancel my subscription",
        "Refund has not been processed yet"
    ]
}

df = pd.DataFrame(data)
df

,ticket
0,I cannot log into my account
1,Payment was deducted but order not confirmed
2,App crashes when I open settings
3,How can I reset my password?
4,The website is very slow today
5,I want to cancel my subscription
6,Refund has not been processed yet


In [ ]:
#Define Candidate Labels
candidate_labels = [
    "Login Issue",
    "Payment Problem",
    "Technical Bug",
    "Account Management",
    "Performance Issue",
    "Subscription",
    "Refund"
]

In [ ]:
# ----------------------------------------------------------
# ZERO-SHOT CLASSIFICATION
# ----------------------------------------------------------
# For each support ticket:
# 1. The model compares the ticket text with candidate labels.
# 2. It assigns probability scores to each label.
# 3. We extract the Top 3 most probable labels.
#
# This approach does not require any additional training.
# ----------------------------------------------------------
results = []

for ticket in df["ticket"]:
    output = classifier(ticket, candidate_labels)

    # Extract Top 3 Tags
    top_3 = list(zip(output["labels"][:3], output["scores"][:3]))

#Create Results DataFrame
    results.append({
        "Ticket": ticket,
        "Top 1 Tag": top_3[0][0],
        "Top 1 Score": top_3[0][1],
        "Top 2 Tag": top_3[1][0],
        "Top 2 Score": top_3[1][1],
        "Top 3 Tag": top_3[2][0],
        "Top 3 Score": top_3[2][1]
    })

results_df = pd.DataFrame(results)
results_df

,Ticket,Top 1 Tag,Top 1 Score,Top 2 Tag,Top 2 Score,Top 3 Tag,Top 3 Score
0,I cannot log into my account,Login Issue,0.769011,Subscription,0.071466,Performance Issue,0.049645
1,Payment was deducted but order not confirmed,Payment Problem,0.494222,Performance Issue,0.129768,Account Management,0.122913
2,App crashes when I open settings,Performance Issue,0.576157,Technical Bug,0.158691,Login Issue,0.111890
3,How can I reset my password?,Login Issue,0.487662,Account Management,0.201227,Subscription,0.093433
4,The website is very slow today,Performance Issue,0.714518,Technical Bug,0.065745,Subscription,0.062441
5,I want to cancel my subscription,Subscription,0.609616,Refund,0.175178,Payment Problem,0.074870
6,Refund has not been processed yet,Refund,0.630402,Payment Problem,0.179828,Subscription,0.052611


In [ ]:
#Add True Labels
true_labels = [
    "Login Issue",
    "Payment Problem",
    "Technical Bug",
    "Account Management",
    "Performance Issue",
    "Subscription",
    "Refund"
]

df["True Label"] = true_labels
df

,ticket,True Label
0,I cannot log into my account,Login Issue
1,Payment was deducted but order not confirmed,Payment Problem
2,App crashes when I open settings,Technical Bug
3,How can I reset my password?,Account Management
4,The website is very slow today,Performance Issue
5,I want to cancel my subscription,Subscription
6,Refund has not been processed yet,Refund


In [ ]:
predicted_labels = results_df["Top 1 Tag"]

df["Predicted Label"] = predicted_labels
df

,ticket,True Label,Predicted Label
0,I cannot log into my account,Login Issue,Login Issue
1,Payment was deducted but order not confirmed,Payment Problem,Payment Problem
2,App crashes when I open settings,Technical Bug,Performance Issue
3,How can I reset my password?,Account Management,Login Issue
4,The website is very slow today,Performance Issue,Performance Issue
5,I want to cancel my subscription,Subscription,Subscription
6,Refund has not been processed yet,Refund,Refund


In [ ]:
#Calculate Zero-Shot Accuracy
accuracy = accuracy_score(df["True Label"], df["Predicted Label"])
print("Zero-Shot Accuracy:", accuracy)
# from output we can predict  model correctly predicted 5 out of 7 tickets.

Zero-Shot Accuracy: 0.7142857142857143


In [ ]:
#Enhanced Labels
#Here we improved labels by adding descriptions.
enhanced_labels = [
    "Login Issue: problems related to login, authentication, password errors, unable to access account",

    "Payment Problem: issues with payment failure, money deducted, transaction errors",

    "Technical Bug: app crashes, software bugs, unexpected errors, system malfunction",

    "Account Management: updating profile, changing account settings, password reset",

    "Performance Issue: slow website, lagging app, loading problems",

    "Subscription: cancelling or modifying subscription plans",

    "Refund: refund requests, money not returned, reimbursement issues"
]

In [ ]:
# Run Classification Again
few_shot_results = []

for ticket in df["ticket"]:
    output = classifier(ticket, enhanced_labels)

    top_3 = list(zip(output["labels"][:3], output["scores"][:3]))

    few_shot_results.append({
        "Ticket": ticket,
        "Top 1 Tag": top_3[0][0],
        "Top 2 Tag": top_3[1][0],
        "Top 3 Tag": top_3[2][0]
    })

few_shot_df = pd.DataFrame(few_shot_results)
few_shot_df

,Ticket,Top 1 Tag,Top 2 Tag,Top 3 Tag
0,I cannot log into my account,"Login Issue: problems related to login, authen...","Payment Problem: issues with payment failure, ...","Technical Bug: app crashes, software bugs, une..."
1,Payment was deducted but order not confirmed,"Payment Problem: issues with payment failure, ...","Refund: refund requests, money not returned, r...",Subscription: cancelling or modifying subscrip...
2,App crashes when I open settings,"Technical Bug: app crashes, software bugs, une...","Performance Issue: slow website, lagging app, ...","Payment Problem: issues with payment failure, ..."
3,How can I reset my password?,"Login Issue: problems related to login, authen...","Account Management: updating profile, changing...",Subscription: cancelling or modifying subscrip...
4,The website is very slow today,"Performance Issue: slow website, lagging app, ...","Technical Bug: app crashes, software bugs, une...","Login Issue: problems related to login, authen..."
5,I want to cancel my subscription,Subscription: cancelling or modifying subscrip...,"Refund: refund requests, money not returned, r...","Login Issue: problems related to login, authen..."
6,Refund has not been processed yet,"Refund: refund requests, money not returned, r...","Payment Problem: issues with payment failure, ...",Subscription: cancelling or modifying subscrip...


In [ ]:
#Remove Description
few_shot_df["Top 1 Tag"] = few_shot_df["Top 1 Tag"].apply(lambda x: x.split(":")[0])
few_shot_df

,Ticket,Top 1 Tag,Top 2 Tag,Top 3 Tag
0,I cannot log into my account,Login Issue,"Payment Problem: issues with payment failure, ...","Technical Bug: app crashes, software bugs, une..."
1,Payment was deducted but order not confirmed,Payment Problem,"Refund: refund requests, money not returned, r...",Subscription: cancelling or modifying subscrip...
2,App crashes when I open settings,Technical Bug,"Performance Issue: slow website, lagging app, ...","Payment Problem: issues with payment failure, ..."
3,How can I reset my password?,Login Issue,"Account Management: updating profile, changing...",Subscription: cancelling or modifying subscrip...
4,The website is very slow today,Performance Issue,"Technical Bug: app crashes, software bugs, une...","Login Issue: problems related to login, authen..."
5,I want to cancel my subscription,Subscription,"Refund: refund requests, money not returned, r...","Login Issue: problems related to login, authen..."
6,Refund has not been processed yet,Refund,"Payment Problem: issues with payment failure, ...",Subscription: cancelling or modifying subscrip...


In [ ]:
#Few-Shot Accuracy
# Now 6 out of 7 correct.
few_shot_accuracy = accuracy_score(df["True Label"], few_shot_df["Top 1 Tag"])
print("Few-Shot Accuracy:", few_shot_accuracy)

Few-Shot Accuracy: 0.8571428571428571


In [ ]:
comparison = pd.DataFrame({
    "Method": ["Zero-Shot", "Few-Shot (Prompt Engineered)"],
    "Accuracy": [accuracy, few_shot_accuracy]
})

comparison
#So accuracy improved by 14%.

#That clearly shows:

#Prompt engineering improves classification performance.

,Method,Accuracy
0,Zero-Shot,0.714286
1,Few-Shot (Prompt Engineered),0.857143


In [ ]:
improvement = few_shot_accuracy - accuracy
print("Accuracy Improvement:", improvement)
#So accuracy improved by 14%.

#That clearly shows:

#Prompt engineering improves classification performance.

Accuracy Improvement: 0.1428571428571428
